# ex38：数据透视表入门（pivot_table / crosstab）

## 练习目标

学习用 pandas 把**长表**转换成**宽表**，从多个维度观察数据。

我们会用一份奶茶店销售数据，完成：
1. 合并销售表和门店表
2. 用 `pivot_table` 做城市 × 商品的销售额透视
3. 用 `crosstab` 统计城市 × 商品的订单数
4. 处理透视表中的空值
5. 把透视结果导出

---

In [ ]:
import pandas as pd

## 前置知识回顾 ①：什么是数据透视表？

透视表的本质是**重新组织数据**的展示方式。

比如原始数据是长这样（每一行是一笔订单）：

| 城市 | 商品 | 销售额 |
|------|------|--------|
| 上海 | 珍珠奶茶 | 750 |
| 上海 | 芒果冰沙 | 540 |
| 南京 | 珍珠奶茶 | 600 |

透视之后可以变成宽表：

| 城市 | 珍珠奶茶 | 芒果冰沙 |
|------|----------|----------|
| 上海 | 750 | 540 |
| 南京 | 600 | 0 |

这种转换让你一眼看出：
- 每个城市各卖了多少每种商品
- 哪个城市哪种商品是空白（可以填 0）
- 方便做热力图等可视化

## 前置知识回顾 ②：pd.pivot_table 的参数

```python
pd.pivot_table(
    data=df,              # 要透视的 DataFrame
    values="revenue",    # 要聚合计算的列（值）
    index="city",        # 行方向的分组列
    columns="product",   # 列方向的分组列
    aggfunc="sum",       # 聚合方式：sum / mean / count 等
    fill_value=0          # 把透视结果中的 NaN 替换成 0
)
```

### 四个核心参数

| 参数 | 作用 | 示例 |
|------|------|------|
| `values` | 要统计的数值列 | `"revenue"` |
| `index` | 结果表格的**行**是什么 | `"city"` |
| `columns` | 结果表格的**列**是什么 | `"product"` |
| `aggfunc` | 怎么汇总这些值 | `"sum"`、`"mean"`、`"count"` |

### 易错点
- `aggfunc` 默认值是 `"mean"`，如果你要算总销售额，记得改成 `"sum"`
- 如果某个城市没有卖过某种商品，结果对应位置会是 `NaN`，可以用 `fill_value=0` 填充
- `index` 和 `columns` 写反的话，行列会颠倒

## 前置知识回顾 ③：crosstab 是什么？

`pd.crosstab()` 专门用来快速做**频数统计**——也就是统计两个分类变量交叉出现的次数。

```python
pd.crosstab(
    index=df["city"],
    columns=df["product"]
)
```

结果含义：每个城市 × 每种商品，各出现了多少笔订单。

### pivot_table vs crosstab 的区别

| 工具 | 侧重点 | 用法 |
|------|--------|------|
| `pivot_table` | 对某个数值列做聚合 | 需要指定 `values` 和 `aggfunc` |
| `crosstab` | 统计出现的频次 | 默认就是 `count` |

你也可以给 `crosstab` 加 `values` 和 `aggfunc`，但初学者先掌握频数统计就够了。

## 前置知识回顾 ④：透视表的索引问题

`pivot_table` 的结果中，`index` 列会变成**索引**，不是普通列。

如果你要把它变回普通列（比如导出 CSV），需要：

```python
result = result.reset_index()
```

`reset_index()` 会把索引变成一列普通列，方便后续处理。

## 第 1 步：读取数据

读取 `ex38_sales.csv` 和 `ex38_stores.csv`，查看内容和形状。

In [ ]:
sales = pd.read_csv("ex38_sales.csv")
stores = pd.read_csv("ex38_stores.csv")

In [ ]:
# 查看 sales 前几行
sales.head()

In [ ]:
# 查看 stores
stores

## 第 2 步：合并销售表和门店表

和 ex37 一样，用 `pd.merge()` 把两张表按 `store_id` 合并。
合并后新增一列 `revenue`（销售额 = quantity × unit_price）。

In [ ]:
# 合并 sales 和 stores
merged = ...

# 新增 revenue 列
merged["revenue"] = ...

# 查看合并结果
merged.head()

## 第 3 步：城市 × 商品 销售额透视表

任务：用 `pivot_table` 生成如下表格

- 行：`city`（城市）
- 列：`product`（商品）
- 值：`revenue` 的 `sum`
- 空值填充为 0

这样可以看到：每个城市每种商品的总销售额。

In [ ]:
pivot_revenue = pd.pivot_table(
    ...
)
pivot_revenue

## 第 4 步：城市 × 商品 订单数交叉表

任务：用 `pd.crosstab()` 统计每个城市每种商品各有多少笔订单。

提示：`crosstab` 的两个主要参数是 `index` 和 `columns`，分别传入 Series。

In [ ]:
order_count = pd.crosstab(
    ...
)
order_count

## 第 5 步：多角度透视练习

再做两个透视表，巩固理解：

1. **城市 × 日期** 的销售额总和（看每天各城市的销售表现）
2. **门店** 的平均客单价（对 `revenue` 求 `mean`，按 `store_name` 分组）

注意第二个不是透视表，用 `groupby` 就可以。

In [ ]:
# 城市 × 日期 销售额透视
pivot_city_date = pd.pivot_table(
    ...
)
pivot_city_date

In [ ]:
# 各门店平均客单价
store_avg = ...
store_avg

## 第 6 步：导出透视结果

把第 3 步的 `pivot_revenue` 导出为 `ex38_pivot_revenue.csv`。

注意：透视表的 `city` 列目前可能是索引，导出前要不要 `reset_index()`？

In [ ]:
# 导出
...

## 思考题

1. 如果不写 `fill_value=0`，透视表里会出现什么？为什么？
2. `pivot_table` 和 `groupby` 都能做分组统计，什么时候用前者，什么时候用后者？
3. `crosstab` 的结果能不能也加上 `margins=True` 参数？试试看有什么变化。